# Periodic Fourier model in two dimensions (Section 5.1, Table 4)

Reversible diffusion on the torus $\mathbb T^2$ with uniform invariant law and generator
$\mathcal A f=\nabla\cdot(a\nabla f)$, $a(x,y)=1+\beta\cos x\cos y$, $\beta=1/2$, i.e. the SDE
$dX=\nabla a(X)\,dt+\sqrt{2a(X)}\,dW$. Dictionary: the $N_K=(2K+1)^2$ tensor trigonometric modes with
$\max(|k_1|,|k_2|)\le K$ (real orthonormal form $1,\sqrt2\cos(k\cdot x),\sqrt2\sin(k\cdot x)$).
Observable: $\hat f_k\propto(1+|k|^2)^{-4}$, normalized to $\|f\|=1$ ($f\in H^\ell(\mathbb T^2)$ for $\ell<7$).

Since $\mathcal A e_k=-|k|^2e_k-\tfrac{\beta}{4}\sum_{s\in\{(\pm1,\pm1)\}}k\cdot(k+s)\,e_{k+s}$, the Galerkin
matrices and the arrays $\mathcal A\psi$, $\mathcal A^2\psi$ are explicit. The notebook reports, for $T=1$ and
$\Delta t\approx1.5/\|A_K\|$: the errors $\|Q^n\Pi_Kf-\mathcal K^Tf\|$ of SDMD-1, SDMD-R, SDMD-2 and the
Galerkin error with exact matrices; the same errors with empirical matrices from $m$ i.i.d. uniform samples
($\gamma=0$); and the computing times for the matrices. The reference solution is the Fourier--Galerkin
semigroup with cutoff $64$ ($N=16641$), which agrees with the cutoffs $48$ and $80$ to $5\times10^{-13}$.
The sampled results use seed 20261029; the population results are deterministic.

In [1]:
import numpy as np, scipy.sparse as sp, scipy.linalg as sla, time, json
from scipy.sparse.linalg import expm_multiply

rng = np.random.default_rng(20261029)
beta, T = 0.5, 1.0
S = np.array([[1, 1], [1, -1], [-1, 1], [-1, -1]])


def modes(K):
    r = np.arange(-K, K + 1)
    k1, k2 = np.meshgrid(r, r, indexing="ij")
    return np.stack([k1.ravel(), k2.ravel()], axis=1)


def idx_of(K, k1, k2):
    return (k1 + K) * (2 * K + 1) + (k2 + K)


def A_sparse(K):
    """Galerkin matrix of A on V_K in the complex exponential basis (real symmetric)."""
    km = modes(K)
    N = len(km)
    rows, cols, vals = [], [], []
    for j, (k1, k2) in enumerate(km):
        rows.append(j); cols.append(j); vals.append(-float(k1 * k1 + k2 * k2))
        for s in S:
            l1, l2 = k1 + s[0], k2 + s[1]
            if max(abs(l1), abs(l2)) <= K:
                rows.append(idx_of(K, l1, l2)); cols.append(j)
                vals.append(-beta / 4 * float(k1 * l1 + k2 * l2))
    return sp.csr_matrix((vals, (rows, cols)), shape=(N, N))


def sub_index(Kbig, K):
    km = modes(K)
    return idx_of(Kbig, km[:, 0], km[:, 1])


def exact_matrices(K):
    """A_K and B_K = Pi_K A^2|_{V_K}; A e_k has support in V_{K+1}, so B_K = (A_{K+1}^2)[V_K, V_K]."""
    A = A_sparse(K).toarray()
    A1 = A_sparse(K + 1)
    sub = sub_index(K + 1, K)
    B = (A1 @ A1).toarray()[np.ix_(sub, sub)]
    return A, B


def f_coeffs(Kref):
    km = modes(Kref)
    c = (1.0 + (km ** 2).sum(1)) ** (-4.0)
    return c / np.linalg.norm(c)


def reference(Kref):
    f = f_coeffs(Kref)
    return f, expm_multiply(T * A_sparse(Kref), f)


def embed(vec, K, Kref):
    out = np.zeros((2 * Kref + 1) ** 2, dtype=np.result_type(vec, float))
    out[sub_index(Kref, K)] = vec
    return out


def step_count(normA):
    n = int(np.ceil(T * normA / 1.5))
    return n, T / n


def propagate(Q, v, n):
    for _ in range(n):
        v = Q @ v
    return v


def spectral_norm_sym(M):
    return float(np.abs(sla.eigvalsh(M)).max())


# real orthonormal basis: 1, sqrt2 cos(k.x), sqrt2 sin(k.x), k in a half lattice
def half_lattice(K):
    km = modes(K)
    return np.array([(k1, k2) for k1, k2 in km if (k1 > 0) or (k1 == 0 and k2 > 0)])


def real_from_complex_matrix(K):
    """unitary U with phi_real = E_complex @ U"""
    N = (2 * K + 1) ** 2
    H = half_lattice(K)
    U = np.zeros((N, N), dtype=complex)
    U[idx_of(K, 0, 0), 0] = 1.0
    for a, (k1, k2) in enumerate(H):
        ip, im = idx_of(K, k1, k2), idx_of(K, -k1, -k2)
        U[ip, 1 + a] = 1 / np.sqrt(2); U[im, 1 + a] = 1 / np.sqrt(2)
        U[ip, 1 + len(H) + a] = 1 / (np.sqrt(2) * 1j); U[im, 1 + len(H) + a] = -1 / (np.sqrt(2) * 1j)
    return U


def to_real_basis(M, U):
    R = U.conj().T @ M @ U
    assert np.abs(R.imag).max() < 1e-9 * (1 + np.abs(R).max())
    return R.real


def coupling_matrices(K):
    """C1: A e_k (k in V_K) expanded in V_{K+1}; C2: A^2 e_k expanded in V_{K+2}."""
    A1 = A_sparse(K + 1); A2 = A_sparse(K + 2)
    C1 = A1[:, sub_index(K + 1, K)].toarray()
    C2 = (A2 @ A2)[:, sub_index(K + 2, K)].toarray()
    return C1, C2


def real_arrays(x, K, C1, C2, with_second):
    """values of the real dictionary, of A psi and (optionally) of A^2 psi at the sample points x (m x 2)"""
    H = half_lattice(K); nH = len(H)
    def E(Kk):
        return np.exp(1j * (x @ modes(Kk).T))
    EK = E(K)
    hp = idx_of(K, H[:, 0], H[:, 1])
    Psi = np.empty((len(x), 1 + 2 * nH))
    Psi[:, 0] = 1.0
    Psi[:, 1:1 + nH] = np.sqrt(2) * EK[:, hp].real
    Psi[:, 1 + nH:] = np.sqrt(2) * EK[:, hp].imag
    del EK
    def real_image(Ec, C):
        W = Ec @ C
        out = np.empty_like(Psi)
        out[:, 0] = W[:, idx_of(K, 0, 0)].real
        out[:, 1:1 + nH] = np.sqrt(2) * W[:, hp].real
        out[:, 1 + nH:] = np.sqrt(2) * W[:, hp].imag
        return out
    APsi = real_image(E(K + 1), C1)
    A2Psi = real_image(E(K + 2), C2) if with_second else None
    return Psi, APsi, A2Psi


def assemble(x, K, C1, C2, with_second, chunk=10000):
    """empirical matrices G, H (and J) from the samples x, accumulated in chunks"""
    m = len(x); N = (2 * K + 1) ** 2
    G = np.zeros((N, N)); Hm = np.zeros((N, N)); J = np.zeros((N, N)) if with_second else None
    for s in range(0, m, chunk):
        Psi, APsi, A2Psi = real_arrays(x[s:s + chunk], K, C1, C2, with_second)
        G += Psi.T @ Psi; Hm += Psi.T @ APsi
        if with_second:
            J += Psi.T @ A2Psi
    G /= m; Hm /= m
    if with_second:
        J /= m
    return G, Hm, J


def g(x):
    return f"{x:.2g}"


def sci(x, sig=3):
    """LaTeX form with sig significant digits"""
    mant, ex = f"{x:.{sig-1}e}".split("e")
    return f"${mant}\\times10^{{{int(ex)}}}$"


## Reference solution

In [2]:
Kref = 64
f_ref, u_ref = reference(Kref)
for Kc in (48, 80):
    fc, uc = reference(Kc)
    a, b = (u_ref, embed(uc, Kc, Kref)) if Kc < Kref else (embed(u_ref, Kref, Kc), uc)
    print(f"|u_{Kref} - u_{Kc}| = {np.linalg.norm(a - b):.2e}")


|u_64 - u_48| = 5.26e-13


|u_64 - u_80| = 5.27e-13


## Exact Galerkin matrices

In [3]:
exact = []
print(f"{'K':>3} {'N':>5} {'n':>5} {'||A_K||':>9} {'Theta_K':>9} {'SDMD-1':>9} {'SDMD-R':>9} {'SDMD-2':>9} {'Galerkin':>9} {'max||Q||':>9}")
for K in [2, 4, 8, 16, 32]:
    A, B = exact_matrices(K)
    normA = spectral_norm_sym(A)
    Theta = spectral_norm_sym(B - A @ A)
    n, dt = step_count(normA)
    fK = f_ref[sub_index(Kref, K)]
    I = np.eye(len(A))
    Q = {"sdmd1": I + dt * A, "sdmdR": I + dt * A + 0.5 * dt * dt * (A @ A), "sdmd2": I + dt * A + 0.5 * dt * dt * B}
    err = {name: float(np.linalg.norm(embed(propagate(M, fK, n), K, Kref) - u_ref)) for name, M in Q.items()}
    err["galerkin"] = float(np.linalg.norm(embed(expm_multiply(T * sp.csr_matrix(A), fK), K, Kref) - u_ref))
    normQ = max(spectral_norm_sym(M) for M in Q.values())
    exact.append(dict(K=K, N=len(A), n=n, dt=dt, normA=normA, Theta=Theta, errors=err, normQ=normQ))
    print(f"{K:>3} {len(A):>5} {n:>5} {g(normA):>9} {g(Theta):>9} {g(err['sdmd1']):>9} {g(err['sdmdR']):>9} {g(err['sdmd2']):>9} {g(err['galerkin']):>9} {normQ:>9.6f}")

Ks = np.array([e["K"] for e in exact], float)
for name in ["sdmd1", "sdmdR", "sdmd2"]:
    err = np.array([e["errors"][name] for e in exact])
    print(f"fitted slope of {name} against K over the three finest cutoffs: {np.polyfit(np.log(Ks[-3:]), np.log(err[-3:]), 1)[0]:.2f}")


  K     N     n   ||A_K||   Theta_K    SDMD-1    SDMD-R    SDMD-2  Galerkin  max||Q||
  2    25     6         8       4.9    0.0042   0.00075   0.00068   0.00065  1.000000
  4    81    22        33        65    0.0011   3.1e-05     3e-05   2.4e-05  1.000000


  8   289    91   1.4e+02     1e+03   0.00025     1e-06     1e-06   5.9e-08  1.000000


 16  1089   391   5.9e+02   1.6e+04   5.9e-05   5.5e-08   5.5e-08   8.8e-13  1.000000


 32  4225  1692   2.5e+03   2.7e+05   1.4e-05   2.9e-09   2.9e-09   5.3e-13  1.000000
fitted slope of sdmd1 against K over the three finest cutoffs: -2.11
fitted slope of sdmdR against K over the three finest cutoffs: -4.23
fitted slope of sdmd2 against K over the three finest cutoffs: -4.23


## Empirical matrices from $m$ i.i.d. uniform samples ($\gamma=0$)

In [4]:
empirical = []
print(f"{'K':>3} {'N':>5} {'m':>6} {'reps':>4} {'SDMD-1':>9} {'SDMD-R':>9} {'SDMD-2':>9} {'xi_A':>8} {'xi_B':>8} {'cond G':>7}")
for K, m, reps in [(2, 10**4, 10), (4, 10**4, 10), (8, 10**4, 10), (16, 10**4, 5)]:
    A, B = exact_matrices(K)
    U = real_from_complex_matrix(K)
    Ar, Br = to_real_basis(A, U), to_real_basis(B, U)
    C1, C2 = coupling_matrices(K)
    n, dt = step_count(spectral_norm_sym(A))
    fK = f_ref[sub_index(Kref, K)]
    fK_real = (U.conj().T @ fK).real
    I = np.eye(len(A))
    out = {k: [] for k in ["sdmd1", "sdmdR", "sdmd2", "xiA", "xiB", "condG"]}
    for r in range(reps):
        x = rng.uniform(0, 2 * np.pi, size=(m, 2))
        G, Hm, J = assemble(x, K, C1, C2, True)
        Ah = np.linalg.solve(G, Hm); Bh = np.linalg.solve(G, J)
        out["xiA"].append(float(np.linalg.norm(Ah - Ar, 2))); out["xiB"].append(float(np.linalg.norm(Bh - Br, 2)))
        out["condG"].append(float(np.linalg.cond(G)))
        Q = {"sdmd1": I + dt * Ah, "sdmdR": I + dt * Ah + 0.5 * dt * dt * (Ah @ Ah), "sdmd2": I + dt * Ah + 0.5 * dt * dt * Bh}
        for name, M in Q.items():
            out[name].append(float(np.linalg.norm(embed(U @ propagate(M, fK_real, n), K, Kref) - u_ref)))
    mean = {k: float(np.mean(v)) for k, v in out.items()}
    empirical.append(dict(K=K, N=len(A), m=m, reps=reps, n=n, dt=dt, mean=mean))
    print(f"{K:>3} {len(A):>5} {m:>6} {reps:>4} {g(mean['sdmd1']):>9} {g(mean['sdmdR']):>9} {g(mean['sdmd2']):>9} {g(mean['xiA']):>8} {g(mean['xiB']):>8} {g(mean['condG']):>7}")


  K     N      m reps    SDMD-1    SDMD-R    SDMD-2     xi_A     xi_B  cond G


  2    25  10000   10    0.0042   0.00074   0.00069     0.16      3.4     1.2


  4    81  10000   10    0.0011   3.2e-05     3e-05     0.96       67     1.5


  8   289  10000   10   0.00025     1e-06     1e-06      6.6  1.7e+03     2.4


 16  1089  10000    5   5.9e-05   5.5e-08   5.5e-08       54  5.3e+04     7.8


## Computing times for the matrices (best of three runs on one further sample) and for the time stepping

In [5]:
timing = []
print(f"{'K':>3} {'N':>5} {'m':>6} {'t_1,R':>8} {'t_2':>8} {'ratio':>6} {'step(1)':>8} {'step(R)':>8} {'step(2)':>8}")
for K, m in [(2, 10**4), (4, 10**4), (8, 10**4), (16, 10**4)]:
    C1, C2 = coupling_matrices(K)
    x = rng.uniform(0, 2 * np.pi, size=(m, 2))
    t1 = []; t2 = []
    for _ in range(3):
        t0 = time.perf_counter(); assemble(x, K, C1, C2, False); t1.append(time.perf_counter() - t0)
        t0 = time.perf_counter(); G, Hm, J = assemble(x, K, C1, C2, True); t2.append(time.perf_counter() - t0)
    # time stepping with the empirical matrices of the last run: n matrix-vector products per scheme
    Ah = np.linalg.solve(G, Hm); Bh = np.linalg.solve(G, J)
    A, _ = exact_matrices(K)
    n, dt = step_count(spectral_norm_sym(A))
    U = real_from_complex_matrix(K)
    v0 = (U.conj().T @ f_ref[sub_index(Kref, K)]).real
    I = np.eye(len(A))
    Q = {"sdmd1": I + dt * Ah, "sdmdR": I + dt * Ah + 0.5 * dt * dt * (Ah @ Ah), "sdmd2": I + dt * Ah + 0.5 * dt * dt * Bh}
    ts = {}
    for name, M in Q.items():
        best = np.inf
        for _ in range(3):
            t0 = time.perf_counter(); propagate(M, v0, n); best = min(best, time.perf_counter() - t0)
        ts[name] = best
    timing.append(dict(K=K, N=(2 * K + 1) ** 2, m=m, t_1R=min(t1), t_2=min(t2), ts=ts))
    print(f"{K:>3} {(2*K+1)**2:>5} {m:>6} {g(min(t1)):>8} {g(min(t2)):>8} {min(t2)/min(t1):>6.2f} {g(ts['sdmd1']):>8} {g(ts['sdmdR']):>8} {g(ts['sdmd2']):>8}")

# data for the right panel of Figure 1 (read by 5_1_fourier_accuracy_cost.ipynb)
emp = {e["K"]: e for e in empirical}
json.dump([dict(K=e["K"], N=e["N"], err=emp[e["K"]]["mean"], t_1R=e["t_1R"], t_2=e["t_2"], ts=e["ts"]) for e in timing],
          open("results_torus2d.json", "w"), indent=1)
print("wrote results_torus2d.json")


  K     N      m    t_1,R      t_2  ratio  step(1)  step(R)  step(2)


  2    25  10000    0.034    0.072   2.14  1.1e-05  1.1e-05  1.1e-05


  4    81  10000     0.11     0.21   1.92  4.4e-05  4.4e-05  5.1e-05


  8   289  10000      0.5     0.97   1.95   0.0011   0.0011   0.0011


 16  1089  10000      3.7        7   1.91     0.13     0.12     0.12
wrote results_torus2d.json


## Table 4 of the paper (LaTeX rows)

In [6]:
emp = {e["K"]: e for e in empirical}
tim = {e["K"]: e for e in timing}
for e in exact:
    K = e["K"]; E = e["errors"]
    row = [str(K), str(e["N"]), sci(E["galerkin"], 2), sci(E["sdmd1"]), sci(E["sdmdR"]), sci(E["sdmd2"])]
    if K in emp:
        row += [sci(emp[K]["mean"]["sdmdR"]), sci(emp[K]["mean"]["sdmd2"]), f"${g(tim[K]['t_1R'])}$", f"${g(tim[K]['t_2'])}$"]
    else:
        row += ["--"] * 4
    print(" & ".join(row) + r" \\")
print(json.dumps(dict(exact=exact, empirical=empirical, timing=timing), indent=1))


2 & 25 & $6.5\times10^{-4}$ & $4.20\times10^{-3}$ & $7.47\times10^{-4}$ & $6.82\times10^{-4}$ & $7.43\times10^{-4}$ & $6.85\times10^{-4}$ & $0.034$ & $0.072$ \\
4 & 81 & $2.4\times10^{-5}$ & $1.06\times10^{-3}$ & $3.05\times10^{-5}$ & $2.97\times10^{-5}$ & $3.18\times10^{-5}$ & $3.00\times10^{-5}$ & $0.11$ & $0.21$ \\
8 & 289 & $5.9\times10^{-8}$ & $2.53\times10^{-4}$ & $1.03\times10^{-6}$ & $1.03\times10^{-6}$ & $1.04\times10^{-6}$ & $1.03\times10^{-6}$ & $0.5$ & $0.97$ \\
16 & 1089 & $8.8\times10^{-13}$ & $5.87\times10^{-5}$ & $5.52\times10^{-8}$ & $5.52\times10^{-8}$ & $5.52\times10^{-8}$ & $5.52\times10^{-8}$ & $3.7$ & $7$ \\
32 & 4225 & $5.3\times10^{-13}$ & $1.36\times10^{-5}$ & $2.94\times10^{-9}$ & $2.94\times10^{-9}$ & -- & -- & -- & -- \\
{
 "exact": [
  {
   "K": 2,
   "N": 25,
   "n": 6,
   "dt": 0.16666666666666666,
   "normA": 8.041806349920906,
   "Theta": 4.853465431395431,
   "errors": {
    "sdmd1": 0.0042018620027542974,
    "sdmdR": 0.0007471398969809548,
    "sdmd2